# Short-Term Morphological Forecast (2026 P1–2028 P6)

**Best model for the bi-monthly horizon** (report section 7.5): `U-Net + LSTM` (§5.6) with sequence length L=9 and seasonal sin/cos channels.

This stand-alone notebook trains the winning architecture on the complete available record and rolls the prediction forward autoregressively to produce the 18-step forecast (2026 P1 … 2028 P6). It writes the forecast area table, the area-trend figure and the erosion/accretion risk map to `outputs/`.

> Best architecture for the bi-monthly horizon (report §7.3): **U-Net + LSTM**, Setup 2, L=9 -- IoU = 0.7791, Dice = 0.8701.


## 1. Configuration and imports

Environment setup, imports and the forecast configuration.


In [ ]:
# ============================================================================
# Environment, imports and configuration  (best-model final prediction)
# ============================================================================
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_strict_conv_algorithm_picker=false")

import re
import glob
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import tensorflow as tf
tf.get_logger().setLevel("FATAL")

from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import (
    Callback, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,
)
from tensorflow.keras.optimizers import Adam

import rasterio
import cv2
from skimage.transform import resize
from sklearn.metrics import precision_score, recall_score

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(42)
tf.random.set_seed(42)
for _gpu in tf.config.experimental.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        pass

# ---- Configuration ---------------------------------------------------------
DATA_DIR         = os.environ.get("BIMONTHLY_DIR", os.path.join("data", "raw", "bimonthly"))
IMG_HEIGHT       = 256
IMG_WIDTH        = 256
N_COMPONENTS     = 3
BINARY_THRESHOLD = 0.5
LEARNING_RATE    = 1e-4
CLIPNORM         = 1.0
DEFAULT_EPOCHS   = 200
EARLY_STOP_PATIENCE = 20
REDUCE_LR_PATIENCE  = 7
BATCH_SIZE       = 4

RESOLUTION        = "bimonthly"
SEQ_LEN           = 9
MODEL_LABEL       = "U-Net + LSTM"
MODEL_KEY         = "unet_lstm"
T_FORECAST        = 18
FORECAST_LABELS   = ['2026 P1', '2026 P2', '2026 P3', '2026 P4', '2026 P5', '2026 P6', '2027 P1', '2027 P2', '2027 P3', '2027 P4', '2027 P5', '2027 P6', '2028 P1', '2028 P2', '2028 P3', '2028 P4', '2028 P5', '2028 P6']
SEASONAL_CHANNELS = True
OUTPUT_DIR        = os.path.join("outputs", RESOLUTION, "forecast")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Data directory :", DATA_DIR)
print("Best model     :", MODEL_LABEL, "(", MODEL_KEY, ") | L =", SEQ_LEN)
print("Forecast       :", T_FORECAST, "steps ->",
      FORECAST_LABELS[0], "...", FORECAST_LABELS[-1])


## 2. Inlined pipeline (preprocessing, losses, metrics, split)

The shared helpers, copied verbatim from `utils/model_utils.py`.


In [ ]:
# ============================================================================
# Losses and metrics  (thesis equations 5.6-5.9)
# ============================================================================


def dice_coefficient(y_true, y_pred, smooth: float = 1e-6):
    """Soft Dice coefficient (differentiable) used as a training metric."""
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def dice_loss(y_true, y_pred):
    """Region-based loss: ``1 - Dice`` (equation 5.7 of the thesis)."""
    return 1.0 - dice_coefficient(y_true, y_pred)


def bce_loss(y_true, y_pred):
    """Pixel-wise binary cross-entropy (equation 5.8 of the thesis)."""
    return K.mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))


def combined_loss(y_true, y_pred):
    """
    Hybrid objective ``L_BCE + L_Dice`` (equation 5.9).

    BCE gives stable early gradients; Dice sharpens boundaries.  The sum
    balances pixel-level accuracy with region-level overlap, which is the
    configuration used for *all* five architectures.
    """
    return bce_loss(y_true, y_pred) + dice_loss(y_true, y_pred)


def iou_metric(y_true, y_pred, threshold: float = BINARY_THRESHOLD):
    """Binarised Intersection-over-Union as a Keras metric."""
    y_pred_b = K.cast(y_pred > threshold, "float32")
    y_true_b = K.cast(y_true > threshold, "float32")
    intersection = K.sum(y_true_b * y_pred_b)
    union = K.sum(y_true_b) + K.sum(y_pred_b) - intersection
    return (intersection + K.epsilon()) / (union + K.epsilon())


# ============================================================================
# NumPy metrics used at evaluation time
# ============================================================================


def iou_np(y_true: np.ndarray, y_pred: np.ndarray, threshold: float = BINARY_THRESHOLD) -> float:
    yt = (y_true > threshold).astype(np.float32)
    yp = (y_pred > threshold).astype(np.float32)
    inter = np.sum(yt * yp)
    union = np.sum(yt) + np.sum(yp) - inter
    return float(inter / (union + 1e-6))


def dice_np(y_true: np.ndarray, y_pred: np.ndarray, threshold: float = BINARY_THRESHOLD) -> float:
    yt = (y_true > threshold).astype(np.float32)
    yp = (y_pred > threshold).astype(np.float32)
    inter = np.sum(yt * yp)
    return float((2.0 * inter) / (np.sum(yt) + np.sum(yp) + 1e-6))


def calculate_area_difference(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    pixel_area_km2: float,
    threshold: float = BINARY_THRESHOLD,
) -> float:
    """ΔA = A_pred − A_true, in km² (equation 5.6)."""
    true_area = np.sum((y_true > threshold).astype(np.float32)) * pixel_area_km2
    pred_area = np.sum((y_pred > threshold).astype(np.float32)) * pixel_area_km2
    return float(pred_area - true_area)


# ============================================================================
# Data loading and preprocessing
# ============================================================================


def keep_largest_n_components_cv2(mask: np.ndarray, n: int = N_COMPONENTS):
    """
    Retain only the ``n`` largest connected components of a binary mask.

    This removes salt-and-pepper noise and isolated ponds so the network
    focuses on the main river channel (thesis § 5.9).
    """
    binary_mask = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    if num_labels <= 1:
        return np.zeros_like(mask)

    areas = stats[1:, cv2.CC_STAT_AREA]
    sorted_idx = np.argsort(areas)[::-1]
    n_keep = min(n, len(sorted_idx))

    cleaned = np.zeros_like(mask)
    for idx in sorted_idx[:n_keep]:
        cleaned[labels == (idx + 1)] = mask.max()
    return cleaned


def load_and_preprocess_image(
    filepath: str, apply_cleaning: bool = True, n_components: int = N_COMPONENTS
) -> np.ndarray:
    """
    Load a GeoTIFF water mask and turn it into a 256×256 clean binary array.

    Steps: read → optional connected-component cleaning → resize (nearest)
    → threshold at 0.5.
    """
    with rasterio.open(filepath) as src:
        img = src.read(1)
    if apply_cleaning:
        img = keep_largest_n_components_cv2(img, n=n_components)
    resized = resize(
        img,
        (IMG_HEIGHT, IMG_WIDTH),
        mode="constant",
        preserve_range=True,
        anti_aliasing=False,
    )
    return (resized > 0.5).astype(np.float32)


def get_pixel_area_km2(reference_tif: str) -> float:
    """
    Compute the ground area (km²) covered by a single 256×256 pixel.

    Handles both EPSG:4326 (geographic) and projected CRSs.
    """
    with rasterio.open(reference_tif) as src:
        bounds = src.bounds
        if src.crs and src.crs.to_epsg() == 4326:
            centre_lat = (bounds.top + bounds.bottom) / 2
            m_per_deg_lon = 111_320 * np.cos(np.radians(centre_lat))
            width_m = (bounds.right - bounds.left) * m_per_deg_lon
            height_m = (bounds.top - bounds.bottom) * 111_320
            total_km2 = (width_m * height_m) / 1e6
        else:
            total_km2 = ((bounds.right - bounds.left) * (bounds.top - bounds.bottom)) / 1e6
    return total_km2 / (IMG_HEIGHT * IMG_WIDTH)


def build_catalog(data_dir: str, pattern: str = "*.tif") -> pd.DataFrame:
    """
    Scan ``data_dir`` for GeoTIFFs and return a year-sorted DataFrame.

    Columns: ``filepath``, ``filename``, ``year``.
    """
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    records = []
    for filepath in files:
        match = re.search(r"(\d{4})", os.path.basename(filepath))
        if match:
            records.append(
                {
                    "filepath": filepath,
                    "filename": os.path.basename(filepath),
                    "year": int(match.group(1)),
                }
            )
    return pd.DataFrame(records).sort_values("year").reset_index(drop=True)


def load_image_stack(data_dir: str) -> Tuple[List[np.ndarray], List[int]]:
    """Load and preprocess every GeoTIFF in ``data_dir`` into a stack."""
    df = build_catalog(data_dir)
    images, years = [], []
    for _, row in df.iterrows():
        images.append(load_and_preprocess_image(row["filepath"]))
        years.append(row["year"])
    return images, years


# ============================================================================
# Sequence generation  (sliding window, section 5.1)
# ============================================================================


def create_sequences(
    images: Sequence[np.ndarray],
    years: Sequence[int],
    seq_len: int,
    horizon: int = 1,
    stride: int = 1,
):
    """
    Build overlapping ``(X, y)`` windows with a stride of 1.

    Parameters
    ----------
    images   : list of (H, W) binary frames, chronological order
    years    : matching year labels
    seq_len  : number of input frames ``L``
    horizon  : prediction horizon (thesis uses 1)
    stride   : window stride (thesis uses 1)

    Returns
    -------
    X, y, input_years, target_years : np.ndarrays
    """
    X, y, in_years, tgt_years = [], [], [], []
    stop = len(images) - seq_len - horizon + 1
    for i in range(0, stop, stride):
        X.append(images[i : i + seq_len])
        y.append(images[i + seq_len + horizon - 1])
        in_years.append(years[i : i + seq_len])
        tgt_years.append(years[i + seq_len + horizon - 1])
    return (np.array(X), np.array(y), np.array(in_years), np.array(tgt_years))


# ============================================================================
# Strict temporal split  (section 5.2)
# ============================================================================


def prepare_split(X_all, y_all, target_years_all, input_years_all, cutoff_year: int):
    """
    Leakage-proof temporal split.

    Training/validation use samples whose *target* and *latest input* years
    are ≤ ``cutoff_year``; testing uses targets strictly after the cutoff.
    The last 15 % of the (chronologically ordered) training samples become
    the validation set.

    Returns
    -------
    X_tr, y_tr, X_val, y_val, X_test, y_test, target_years_test
    (all arrays already expanded with a trailing channel axis)
    """
    max_input_year = np.array([iy.max() for iy in input_years_all])

    train_mask = (target_years_all <= cutoff_year) & (max_input_year <= cutoff_year)
    test_mask = target_years_all > cutoff_year

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    ty_train = target_years_all[train_mask]
    X_test, y_test = X_all[test_mask], y_all[test_mask]
    ty_test = target_years_all[test_mask]

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(
            f"Empty split! Train={len(X_train)}, Test={len(X_test)} " f"(cutoff={cutoff_year})"
        )

    order = np.argsort(ty_train)
    X_train, y_train, ty_train = X_train[order], y_train[order], ty_train[order]

    split = int(len(X_train) * 0.85)
    X_tr, y_tr = X_train[:split], y_train[:split]
    X_val, y_val = X_train[split:], y_train[split:]

    if len(X_val) == 0:
        raise ValueError("Empty validation set!")

    overlap = set(ty_train.tolist()) & set(ty_test.tolist())
    assert not overlap, f"DATA LEAK! Overlapping years: {overlap}"

    def _add_channel(a):
        return np.expand_dims(a, axis=-1)

    return (
        _add_channel(X_tr),
        _add_channel(y_tr),
        _add_channel(X_val),
        _add_channel(y_val),
        _add_channel(X_test),
        _add_channel(y_test),
        ty_test,
    )


# ============================================================================
# Evaluation helpers
# ============================================================================


def evaluate_model(
    model,
    X_test,
    y_test,
    target_years_test,
    pixel_area_km2: float,
    model_label: str,
    setup_name: str,
    include_persistence: bool = True,
) -> pd.DataFrame:
    """
    Evaluate a trained model (and optionally a persistence baseline) on the
    test set, returning a tidy per-sample DataFrame with IoU, Dice,
    Precision, Recall and signed area difference.
    """
    pred = model.predict(X_test, verbose=0)
    comparisons = [(model_label, pred)]
    if include_persistence:
        comparisons.append(("Persistence", X_test[:, -1, :, :, :]))

    rows = []
    for name, predictions in comparisons:
        for i in range(len(y_test)):
            yt = y_test[i, :, :, 0]
            yp = predictions[i, :, :, 0]
            yt_flat = (yt.flatten() > BINARY_THRESHOLD).astype(int)
            yp_flat = (yp.flatten() > BINARY_THRESHOLD).astype(int)
            rows.append(
                {
                    "Setup": setup_name,
                    "Model": name,
                    "Year": target_years_test[i],
                    "IoU": iou_np(yt, yp),
                    "Dice": dice_np(yt, yp),
                    "Precision": precision_score(yt_flat, yp_flat, zero_division=0),
                    "Recall": recall_score(yt_flat, yp_flat, zero_division=0),
                    "Area_Diff_km2": calculate_area_difference(yt, yp, pixel_area_km2),
                }
            )
    return pd.DataFrame(rows)


def summarise_results(df_results: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-sample results into per (Setup, Model) means."""
    cols = ["IoU", "Dice", "Precision", "Recall", "Area_Diff_km2"]
    agg = df_results.groupby(["Setup", "Model"])[cols].mean().round(4)
    agg["Abs_Area_Diff_km2"] = (
        df_results.groupby(["Setup", "Model"])["Area_Diff_km2"]
        .apply(lambda s: s.abs().mean())
        .round(4)
    )
    return agg.reset_index()


## 3. Load the full bi-monthly water-mask record


In [ ]:
# ============================================================================
# Load the full bi-monthly water-mask record
# ============================================================================
images, years = load_image_stack(DATA_DIR)
print(f"Loaded {len(images)} bi-month frames: {years[0]} ... {years[-1]}")
print("Frame shape :", images[0].shape)

PIXEL_AREA_KM2 = get_pixel_area_km2(build_catalog(DATA_DIR).iloc[0]["filepath"])
print(f"Pixel ground area : {PIXEL_AREA_KM2:.6f} km2")


## 4. Best model architecture

The U-Net + LSTM network, inlined.


In [ ]:
# ============================================================================
# Optimiser  (thesis section 5.9)
# ============================================================================


def _adam():
    return Adam(learning_rate=LEARNING_RATE, clipnorm=CLIPNORM)


def _conv_bn(x, filters, dropout=0.0):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    if dropout:
        x = layers.Dropout(dropout)(x)
    return x


def build_unet_lstm(seq_len: int, input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)):
    """
    Architecture 2 – U-Net with a ConvLSTM bottleneck (thesis Figure 5.2).

    The encoder is applied per-frame (TimeDistributed); the bottleneck
    models temporal dependencies; the decoder restores resolution using
    skip connections.
    """
    inputs = layers.Input(shape=(seq_len, *input_shape))
    td = layers.TimeDistributed

    # ── Encoder ──────────────────────────────────────────────────────────────
    e1 = td(layers.Conv2D(32, 3, padding="same", activation="relu"))(inputs)
    e1 = td(layers.BatchNormalization())(e1)
    e1 = td(layers.Conv2D(32, 3, padding="same", activation="relu"))(e1)
    e1 = td(layers.BatchNormalization())(e1)
    p1 = td(layers.MaxPooling2D(2))(e1)
    p1 = td(layers.Dropout(0.15))(p1)

    e2 = td(layers.Conv2D(64, 3, padding="same", activation="relu"))(p1)
    e2 = td(layers.BatchNormalization())(e2)
    e2 = td(layers.Conv2D(64, 3, padding="same", activation="relu"))(e2)
    e2 = td(layers.BatchNormalization())(e2)
    p2 = td(layers.MaxPooling2D(2))(e2)
    p2 = td(layers.Dropout(0.2))(p2)

    e3 = td(layers.Conv2D(128, 3, padding="same", activation="relu"))(p2)
    e3 = td(layers.BatchNormalization())(e3)
    e3 = td(layers.Conv2D(128, 3, padding="same", activation="relu"))(e3)
    e3 = td(layers.BatchNormalization())(e3)
    p3 = td(layers.MaxPooling2D(2))(e3)
    p3 = td(layers.Dropout(0.2))(p3)

    # ── Bottleneck (spatial conv + temporal ConvLSTM) ────────────────────────
    b = td(layers.Conv2D(256, 3, padding="same", activation="relu"))(p3)
    b = td(layers.BatchNormalization())(b)
    b = layers.ConvLSTM2D(256, (3, 3), padding="same", return_sequences=True, dropout=0.3)(b)
    b = layers.BatchNormalization()(b)
    b = layers.ConvLSTM2D(128, (3, 3), padding="same", return_sequences=False, dropout=0.3)(b)
    b = layers.BatchNormalization()(b)

    take_last = layers.Lambda(lambda t: t[:, -1, ...], name="take_last_frame")

    # ── Decoder ──────────────────────────────────────────────────────────────
    d3 = layers.UpSampling2D(2)(b)
    d3 = layers.Concatenate()([d3, take_last(e3)])
    d3 = _conv_bn(d3, 128, 0.2)
    d3 = _conv_bn(d3, 128)

    d2 = layers.UpSampling2D(2)(d3)
    d2 = layers.Concatenate()([d2, take_last(e2)])
    d2 = _conv_bn(d2, 64, 0.2)
    d2 = _conv_bn(d2, 64)

    d1 = layers.UpSampling2D(2)(d2)
    d1 = layers.Concatenate()([d1, take_last(e1)])
    d1 = _conv_bn(d1, 32)

    d1 = layers.Conv2D(16, 3, padding="same", activation="relu")(d1)
    outputs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(d1)

    model = models.Model(inputs, outputs, name=f"UNetLSTM_seq{seq_len}")
    model.compile(optimizer=_adam(), loss=combined_loss, metrics=[dice_coefficient, iou_metric])
    return model


## 5. Train on the full record and forecast autoregressively


In [ ]:
# ============================================================================
# Train the best model on the full record and roll the forecast forward
# ============================================================================
class StructuredTrainingLogger(Callback):
    """Compact, aligned one-line-per-epoch training logger."""

    def __init__(self, total_epochs: int):
        super().__init__()
        self.total_epochs = total_epochs
        self.best_val_loss = np.inf

    def on_train_begin(self, logs=None):
        header = (
            f"{'Ep':>4s}/{'Tot':<4s} | {'Loss':>8s} | {'VLoss':>8s} | "
            f"{'Dice':>6s} | {'VDice':>6s} | {'IoU':>6s} | "
            f"{'VIoU':>6s} | {'LR':>9s} | Note"
        )
        print("-" * len(header))
        print(header)
        print("-" * len(header))

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        vloss = logs.get("val_loss", 0)
        lr = float(K.get_value(self.model.optimizer.learning_rate))
        note = ""
        if vloss < self.best_val_loss:
            self.best_val_loss = vloss
            note = "saved"
        print(
            f"{epoch + 1:>4d}/{self.total_epochs:<4d} | "
            f"{logs.get('loss', 0):>8.4f} | {vloss:>8.4f} | "
            f"{logs.get('dice_coefficient', 0):>6.4f} | "
            f"{logs.get('val_dice_coefficient', 0):>6.4f} | "
            f"{logs.get('iou_metric', 0):>6.4f} | "
            f"{logs.get('val_iou_metric', 0):>6.4f} | {lr:>9.2e} | {note}"
        )

    def on_train_end(self, logs=None):
        print("-" * 85)
        print(f"  Training finished  |  Best val_loss: {self.best_val_loss:.4f}")
        print("-" * 85)


def create_callbacks(
    model_name: str,
    checkpoint_dir: str = "checkpoints",
    epochs: int = DEFAULT_EPOCHS,
    patience: int = EARLY_STOP_PATIENCE,
):
    """Standard callback set (checkpoint, early stop, LR schedule, logger)."""
    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_dir, f"{model_name}_best.keras")
    return [
        ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, mode="min", verbose=0),
        EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=REDUCE_LR_PATIENCE, min_lr=1e-7, verbose=0
        ),
        StructuredTrainingLogger(total_epochs=epochs),
    ]


N_CHANNELS = 3 if SEASONAL_CHANNELS else 1


def to_cube(frames):
    """Stack frames into a (T, H, W, C) tensor, optionally adding sin/cos
    channels that encode the position inside the annual cycle."""
    stack = np.stack(frames, axis=0).astype(np.float32)
    if not SEASONAL_CHANNELS:
        return np.expand_dims(stack, -1)
    T, H, W = stack.shape
    cube = np.empty((T, H, W, 3), dtype=np.float32)
    for t in range(T):
        phase = 2.0 * np.pi * ((t % 6) / 6.0)
        cube[t, ..., 0] = stack[t]
        cube[t, ..., 1] = np.sin(phase)
        cube[t, ..., 2] = np.cos(phase)
    return cube


def make_sequences(cube, seq_len, horizon=1):
    """Sliding-window sequences (stride 1); the target is the next frame."""
    X, y = [], []
    for i in range(len(cube) - seq_len - horizon + 1):
        X.append(cube[i:i + seq_len])
        y.append(cube[i + seq_len + horizon - 1, ..., 0])
    return np.array(X), np.expand_dims(np.array(y), -1)


cube = to_cube(images)
X, y = make_sequences(cube, SEQ_LEN)
print("Training windows :", X.shape)

tf.keras.backend.clear_session()
model = build_unet_lstm(SEQ_LEN, input_shape=(IMG_HEIGHT, IMG_WIDTH, N_CHANNELS))
model.fit(X, y, epochs=DEFAULT_EPOCHS, batch_size=BATCH_SIZE,
          callbacks=create_callbacks(MODEL_KEY, checkpoint_dir=OUTPUT_DIR),
          verbose=0)

# ---- Autoregressive roll-out -----------------------------------------------
window = cube[-SEQ_LEN:][None, ...]
baseline = float((images[-1] > BINARY_THRESHOLD).sum()) * PIXEL_AREA_KM2
areas, masks = [], []

for step in range(T_FORECAST):
    pred = model.predict(window, verbose=0)[0, :, :, 0]
    mask = (pred > BINARY_THRESHOLD).astype(np.float32)
    masks.append(mask)
    areas.append(float(mask.sum()) * PIXEL_AREA_KM2)

    if SEASONAL_CHANNELS:
        phase = 2.0 * np.pi * (((len(images) + step) % 6) / 6.0)
        H, W = pred.shape
        nxt = np.stack([pred,
                        np.full((H, W), np.sin(phase), dtype=np.float32),
                        np.full((H, W), np.cos(phase), dtype=np.float32)],
                       axis=-1)[None, ...]
    else:
        nxt = pred[None, ..., None]
    window = np.concatenate([window[:, 1:, ...], nxt], axis=1)

masks = np.stack(masks, axis=0)
df_fc = pd.DataFrame({"Period": FORECAST_LABELS, "Area_km2": np.round(areas, 2)})
df_fc["Delta_km2"] = np.round(df_fc["Area_km2"] - baseline, 2)
df_fc.to_csv(os.path.join(OUTPUT_DIR, "forecast_areas.csv"), index=False)

print()
print(df_fc.to_string(index=False))
print()
print(f"Baseline (last observed) : {baseline:,.2f} km2")
print(f"Final forecast           : {areas[-1]:,.2f} km2")
print(f"Net change               : {areas[-1] - baseline:+,.2f} km2 "
      f"({100 * (areas[-1] - baseline) / baseline:+.1f}%)")


## 6. Forecast figures and risk map


In [ ]:
# ============================================================================
# Forecast area trend, change-frequency (risk) map and sample masks
# ============================================================================
# -- 1. Forecast water-area trend --------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(areas)), areas, marker="o", color="tab:blue")
ax.axhline(baseline, color="grey", ls="--", label="baseline (last observed)")
ax.set_xticks(range(len(areas)))
ax.set_xticklabels(FORECAST_LABELS, rotation=90)
ax.set_ylabel("water area (km²)")
ax.set_title(f"{RESOLUTION} forecast -- {MODEL_LABEL} (L={SEQ_LEN})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "forecast_area_trend.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# -- 2. Water-occurrence frequency (erosion / accretion risk map) ------------
freq = masks.mean(axis=0)
fig, ax = plt.subplots(figsize=(6.5, 6))
im = ax.imshow(freq, cmap="viridis", vmin=0, vmax=1)
ax.set_title("Forecast water-occurrence frequency (risk map)")
ax.axis("off")
plt.colorbar(im, fraction=0.046, shrink=0.85)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "forecast_risk_map.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# -- 3. A few forecast masks -------------------------------------------------
n_show = min(6, T_FORECAST)
idx = np.linspace(0, T_FORECAST - 1, n_show, dtype=int)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3.4))
for ax, i in zip(np.atleast_1d(axes), idx):
    ax.imshow(masks[i], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(FORECAST_LABELS[i], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "forecast_samples.png"),
            dpi=150, bbox_inches="tight")
plt.show()
